In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from toponymy.annotation import NodeId, AnnotationTree, Annotation, AnnotationStore, Executor

05:18:37 - LiteLLM:WARNING: common_utils.py:979 - litellm: could not pre-load bedrock-runtime response stream shape — Bedrock event-stream decoding will be unavailable. Error: No module named 'botocore'
05:18:37 - LiteLLM:WARNING: common_utils.py:24 - litellm: could not pre-load sagemaker-runtime response stream shape — SageMaker event-stream decoding will be unavailable. Error: No module named 'botocore'


In [3]:
import numpy as np

In [4]:
from toponymy.tools.notebook_data_load import load_small_newsgroups
newsgroups_df = load_small_newsgroups()
embeddings = np.stack(newsgroups_df["embedding"].values)
document_map = np.stack(newsgroups_df["map"].values)

In [5]:
from toponymy import ToponymyClusterer
clusterer = ToponymyClusterer(min_clusters=4, verbose=True)
clusterer.fit(document_map, embeddings);

Layer 0 found 8 clusters
Layer 1 found 4 clusters


In [6]:
clusterer.cluster_tree_

{(1, 0): [(0, 0)],
 (1, 1): [(0, 1)],
 (1, 2): [(0, 7), (0, 6), (0, 5)],
 (1, 3): [(0, 4)],
 (2, 0): [(1, 0), (1, 1), (1, 2), (1, 3), (0, 2), (0, 3)]}

In [7]:
node = NodeId(1, 0)

In [8]:
node.layer

1

In [9]:
node.cluster

0

In [10]:
node == (1, 0)

True

In [11]:
tree = AnnotationTree.from_clusterer(clusterer)

In [12]:
objects = newsgroups_df["post"].str.strip().values
embedding_vectors = embeddings
clusterable_vectors = document_map

## Exemplar Annotation


Need:
* centroid annotation
* cluster objects annotation
* cluster object vectors annotation

In [13]:
centroid_vectors_layered_list = [clusterer.cluster_layers_[0].centroid_vectors, clusterer.cluster_layers_[1].centroid_vectors]

In [14]:
centroid_annotation = Annotation.from_layered_list("cluster_centroid", tree, centroid_vectors_layered_list)

In [15]:
from toponymy.exemplar_texts import diverse_exemplars_by_cluster

In [16]:
cluster_label_vectors_list = [clusterer.cluster_layers_[0].cluster_labels, clusterer.cluster_layers_[1].cluster_labels]

### Create `cluster_object_annotation` from `objects`

In [17]:
object_list = newsgroups_df["post"].str.strip().values

In [18]:
#def from_cluster_list(objects, cluster_label_vectors_list):
cluster_object_annotation = Annotation("cluster_objects", tree)
for layer_id, cluster_label_vector in enumerate(cluster_label_vectors_list):
    for cluster_id in range(cluster_label_vector.max() + 1):
        cluster_mask = cluster_label_vector == cluster_id
        original_indices = np.where(cluster_mask)[0]
        cluster_object_annotation[layer_id, cluster_id] = [object_list[i] for i in original_indices]    

In [19]:
cluster_object_annotation.states

{NodeId(0, 0): <AnnotationState.COMPUTED: 'computed'>,
 NodeId(0, 1): <AnnotationState.COMPUTED: 'computed'>,
 NodeId(0, 2): <AnnotationState.COMPUTED: 'computed'>,
 NodeId(0, 3): <AnnotationState.COMPUTED: 'computed'>,
 NodeId(0, 4): <AnnotationState.COMPUTED: 'computed'>,
 NodeId(0, 5): <AnnotationState.COMPUTED: 'computed'>,
 NodeId(0, 6): <AnnotationState.COMPUTED: 'computed'>,
 NodeId(0, 7): <AnnotationState.COMPUTED: 'computed'>,
 NodeId(1, 0): <AnnotationState.COMPUTED: 'computed'>,
 NodeId(1, 1): <AnnotationState.COMPUTED: 'computed'>,
 NodeId(1, 2): <AnnotationState.COMPUTED: 'computed'>,
 NodeId(1, 3): <AnnotationState.COMPUTED: 'computed'>}

### Create `cluster_object_vectors_annotation` from `object_vectors`

In [20]:
object_vectors = embedding_vectors

In [21]:
cluster_object_vectors_annotation = Annotation("cluster_object_vectors", tree)
null_topic = np.mean(object_vectors, axis=0)

for layer_id, cluster_label_vector in enumerate(cluster_label_vectors_list):
    for cluster_id in range(cluster_label_vector.max() + 1):
        cluster_mask = cluster_label_vector == cluster_id
        cluster_object_vectors_annotation[layer_id, cluster_id] = object_vectors[cluster_mask] - null_topic

In [22]:
cluster_object_vectors_annotation.states

{NodeId(0, 0): <AnnotationState.COMPUTED: 'computed'>,
 NodeId(0, 1): <AnnotationState.COMPUTED: 'computed'>,
 NodeId(0, 2): <AnnotationState.COMPUTED: 'computed'>,
 NodeId(0, 3): <AnnotationState.COMPUTED: 'computed'>,
 NodeId(0, 4): <AnnotationState.COMPUTED: 'computed'>,
 NodeId(0, 5): <AnnotationState.COMPUTED: 'computed'>,
 NodeId(0, 6): <AnnotationState.COMPUTED: 'computed'>,
 NodeId(0, 7): <AnnotationState.COMPUTED: 'computed'>,
 NodeId(1, 0): <AnnotationState.COMPUTED: 'computed'>,
 NodeId(1, 1): <AnnotationState.COMPUTED: 'computed'>,
 NodeId(1, 2): <AnnotationState.COMPUTED: 'computed'>,
 NodeId(1, 3): <AnnotationState.COMPUTED: 'computed'>}

In [23]:
cluster_object_vectors_annotation[0,0]

array([[-0.03516523, -0.02470507, -0.00457667, ..., -0.00025323,
         0.00974339, -0.01337929],
       [-0.02701342,  0.02090386, -0.0088631 , ..., -0.02663631,
        -0.04598448, -0.02278181],
       [-0.04193112,  0.03203582, -0.00346031, ...,  0.0042168 ,
         0.05402277, -0.01278228],
       ...,
       [-0.01031948, -0.05649849, -0.03335569, ...,  0.03093364,
         0.06822366,  0.0142484 ],
       [-0.03946716, -0.04731105, -0.03005292, ..., -0.03029699,
         0.00874201, -0.01366989],
       [-0.02024932,  0.0785161 ,  0.00673887, ...,  0.08301054,
         0.02758984, -0.01715851]], shape=(20, 768))

## AnnotationStore

**TODO**: Add `AnnotationStore.from_clusterer(clusterer, objects, object_vectors)` constructor helper that returns the following store

In [24]:
store = AnnotationStore(tree, [centroid_annotation, cluster_object_annotation, cluster_object_vectors_annotation])

In [25]:
executor = Executor(store)

## Compute exemplars

raw version

In [26]:
store

AnnotationStore(cluster_centroid[12/12], cluster_objects[12/12], cluster_object_vectors[12/12])

In [27]:
layer = clusterer.cluster_layers_[0]

In [28]:
null_topic = np.mean(object_vectors, axis=0)

In [29]:
class DiverseExemplarAnnotatorNotebook:
    inputs = ("cluster_objects", "cluster_centroid", "cluster_object_vectors",)
    outputs = ("exemplars", "examplar_original_indices")
    algorithm_type = "node-node"

    def __init__(self, null_topic=None, n_exemplars=None, diversify_alpha=None, object_to_text_function=None, cluster_label_vectors_list=None):
        self.null_topic=null_topic
        self.n_exemplars=n_exemplars
        self.diversify_alpha=diversify_alpha
        self.object_to_text_function=object_to_text_function
        self.cluster_label_vectors_list=cluster_label_vectors_list
        print(self.n_exemplars)

    def annotate(
        self,
        node,
        *,
        cluster_objects,
        cluster_centroid,
        cluster_object_vectors,# has to match inputs name
    ):

        node_result = diverse_exemplars_by_cluster(cluster_objects=cluster_objects,
                                                   cluster_centroid=cluster_centroid,
                                                   cluster_object_vectors=cluster_object_vectors, 
                                                   null_topic=self.null_topic,
                                                   n_exemplars=self.n_exemplars,
                                                   diversify_alpha=self.diversify_alpha,
                                                   object_to_text_function=self.object_to_text_function)
        chosen_exemplars, exemplar_order, chosen_indices  = node_result
        cluster_mask = cluster_label_vectors_list[node.layer] == node.cluster
        original_indices = np.where(cluster_mask)[0]
        chosen_original_indices = [
            original_indices[exemplar_order[i]] for i in chosen_indices
        ]
        
        return {"exemplars" : chosen_exemplars, "examplar_original_indices": original_indices} # keys have to match outputs

In [30]:
from toponymy.exemplar_texts import DiverseExemplarAnnotator

In [31]:
#%debug
failures = executor.run(DiverseExemplarAnnotatorNotebook(null_topic=null_topic,
                                      n_exemplars=layer.n_exemplars,
                                      diversify_alpha=layer.exemplars_diversify_alpha, 
                                      object_to_text_function=layer.object_to_text_function, 
                                      cluster_label_vectors_list=cluster_label_vectors_list))

8


In [32]:
executor.store

AnnotationStore(cluster_centroid[12/12], cluster_objects[12/12], cluster_object_vectors[12/12], exemplars[12/12], examplar_original_indices[12/12])

In [33]:
executor.store.exemplars[0,0]

['Well, it looks like, just as Doug trumped Tim, beating him to the net\nwith his defensive analyses, so Tim has gotten in ahead of me.\n\nThe way I was doing it was a little different. Being me, of course, I\nused equivalent averages to work out how many runs a player was worth,\nand I calculated both rate of performance (fielding equivalent\naverage) and total performance (fielding equivalent runs). But I\ncompared, not to the average player, but the replacement player, and\nhere\'s why: because the positional adjustment comes built in to the\nsystem. In the AL of 1992, the average SS is 32.9 runs above\nreplacement (RAR); cf, 31.6; 2B, 28.8; 3B, 26.3; LF, 26.0; RF, 24.6;\n1B, 16.9. We may quibble with the exact numbers, but the order looks\nsubstantially right.\n\nIn the equivalent average, I have always set league average to .235. I\nhad decided in hitting that the replacement level batter has an eqa of\n.180; the name of that replacement level hitter, often as not, is\n"Billy Ripk

## Topic Annotator

In [34]:
from toponymy.prompt_construction import topic_name_prompt, topic_name_prompt_by_node

In [35]:
node = NodeId(0, 0)

In [36]:
node

NodeId(0, 0)

In [37]:
from toponymy.prompt_construction import SUMMARY_KINDS

In [38]:
def summary_kind(node, lowest_detail_level=0.0, highest_detail_level=1.0):
    detail_levels = np.linspace(
        lowest_detail_level,
        highest_detail_level,
        tree.n_layers
    )
    summary_level = int(round(detail_levels[node.layer] * (len(SUMMARY_KINDS) - 1)))
    summary_kind = SUMMARY_KINDS[summary_level]
    return summary_kind

In [39]:
prompt = topic_name_prompt_by_node(
        node=node,
        major_subtopics=[],
        minor_subtopics=[],
        other_subtopics=[],
        current_exemplars=store.exemplars[node],
        current_keyphrases=[],
        object_description="newsgroup posts",
        corpus_description="20-newsgroups dataset",
        summary_kind=summary_kind(node),
        exemplar_start_delimiter="<EXAMPLE_POST>\n",
        exemplar_end_delimiter="\n</EXAMPLE_POST>\n\n",
    )

In [40]:
from toponymy.templates import GET_TOPIC_NAME_REGEX
from toponymy.annotation import DescendantsInput

currently `TopicNameAnnotator` handles the subtopics computed from descendants in the tree but not yet the `make_subtopics` created from the bottom layer of topics. That will come, as the input/output specs to give the executor instructions needed to compute Annotators by layer at time still needs to be added. 

In [41]:
class TopicNameAnnotator:
    inputs = []
    optional_inputs = ["exemplars", "output_topics", "subtopics", "keyphrases"] # use it if it's there. don't if not.
    outputs = ["topics"]
    algorithm_type = {"exemplars": ("node", "node"),
                      "output_topics": (DescendantsInput(from_output="topics"), "node"),
                      "subtopics": ("node", "node"),
                      "keyphrases": ("node", "node")}

    def __init__(self, llm_namer=None, prompt_template=None):
        self.llm=llm_namer
        self.prompt_template=prompt_template

    def annotate(
        self,
        node,
        *,
        exemplars=None,
        output_topics=None,
        subtopics=None,
        keyphrases=None,
        
    ):
        if exemplars is None: 
            exemplars = []
        if keyphrases is None:
            keyphrases = []

        if (output_topics is not None) and (len(output_topics) > 0): # slightly different logic here still
            if len(output_topics) == 1 and (list(output_topics.values())[0] != ""):
                print(f"{node}: returning")
                return {"topics" : list(output_topics.values())[0]}

            major_subtopics = [output_topic for child, output_topic in output_topics.items() if child.layer == node.layer - 1]
            minor_subtopics = [output_topic for child, output_topic in output_topics.items() if child.layer == node.layer - 2]

            # promote minor_subtopics to major_subtopics if there are not enough children one layer down    
            if len(major_subtopics) <= 1:
                major_subtopics = major_subtopics + minor_subtopics
                minor_subtopics = [output_topic for child, output_topic in output_topics.items() if child.layer < node.layer - 2]
    
            if node.layer > 1:
                other_subtopics = subtopics
            else:
                other_subtopics = []
            print(node)
            print(major_subtopics)
            print(minor_subtopics)
            print(other_subtopics)
        else:
            major_subtopics = []
            minor_subtopics = []
            other_subtopics = []
            

        prompt = topic_name_prompt_by_node(
                node=node,
                major_subtopics=major_subtopics,
                minor_subtopics=minor_subtopics,
                other_subtopics=other_subtopics,
                current_exemplars=exemplars,
                current_keyphrases=keyphrases,
                object_description="newsgroup posts",
                corpus_description="20-newsgroups dataset",
                summary_kind=summary_kind(node),
                exemplar_start_delimiter="<EXAMPLE_POST>\n",
                exemplar_end_delimiter="\n</EXAMPLE_POST>\n\n",
            )
        if isinstance(prompt, dict) or not prompt.startswith("[!SKIP!]: "):
            topic_name = self.llm.generate_topic_name(
                prompt,
                topic_extraction_function=(
                    self.prompt_template["layer"]["extract_topic_name"]
                    if self.prompt_template
                    else lambda json_response: str(json_response["topic_name"])
                ),
                get_topic_name_regex=(
                    self.prompt_template["layer"].get(
                        "get_topic_name_regex", GET_TOPIC_NAME_REGEX
                    )
                    if self.prompt_template
                    else GET_TOPIC_NAME_REGEX
                ),
            )
        else:
            topic_name = prompt.removeprefix("[!SKIP!]: ")
        if topic_name == '':
            raise ValueError("Failed to generate a valid topic name")
        return {"topics" : topic_name} # keys have to match outputs

In [42]:
from toponymy.llm_wrappers import OllamaNamer
from toponymy.debug_logging import BasicDebugLogger
from toponymy.tools.notebook_data_load import notebook_output_dir

debug_logger_file = notebook_output_dir() / "llm_debug_topic_annotation.jsonl"
debug_logger_file.write_text("", encoding="utf-8")  # Clear the file before starting
debug_logger = BasicDebugLogger(debug_logger_file, truncate=True)

namer = OllamaNamer(callback=debug_logger)
namer.test_llm_connectivity()  # Verify connection
namer.connectivity_status()

{'success': True,
 'model': 'ollama_chat/llama3.2',
 'wrapper': 'LiteLLMNamer',
 'response': '{"status": "ok"}',
 'error_type': None,
 'error_message': None,
 'original_exception': None}

In [43]:
executor.store

AnnotationStore(cluster_centroid[12/12], cluster_objects[12/12], cluster_object_vectors[12/12], exemplars[12/12], examplar_original_indices[12/12])

In [ ]:
failures = executor.run(TopicNameAnnotator(llm_namer=namer, prompt_template=layer.prompt_template))

In [45]:
failures

{NodeId(0, 0): 'ValueError: Failed to generate a valid topic name',
 NodeId(0, 3): 'ValueError: Failed to generate a valid topic name',
 NodeId(0, 6): 'ValueError: Failed to generate a valid topic name'}

In [46]:
store.topics.states

{NodeId(0, 0): <AnnotationState.FAILED: 'failed'>,
 NodeId(0, 1): <AnnotationState.COMPUTED: 'computed'>,
 NodeId(0, 2): <AnnotationState.COMPUTED: 'computed'>,
 NodeId(0, 3): <AnnotationState.FAILED: 'failed'>,
 NodeId(0, 4): <AnnotationState.COMPUTED: 'computed'>,
 NodeId(0, 5): <AnnotationState.COMPUTED: 'computed'>,
 NodeId(0, 6): <AnnotationState.FAILED: 'failed'>,
 NodeId(0, 7): <AnnotationState.COMPUTED: 'computed'>,
 NodeId(1, 0): <AnnotationState.COMPUTED: 'computed'>,
 NodeId(1, 1): <AnnotationState.COMPUTED: 'computed'>,
 NodeId(1, 2): <AnnotationState.COMPUTED: 'computed'>,
 NodeId(1, 3): <AnnotationState.COMPUTED: 'computed'>}

In [47]:
from toponymy.annotation import AnnotationState

In [48]:
for node in tree.nodes:
    if store.topics.states[node] == AnnotationState.COMPUTED:
        print(node, store.topics[node])
    else:
        print(node, store.topics.states[node])

(0, 0) AnnotationState.FAILED
(0, 1) Computer Hardware and Graphics
(0, 2) Cryptography
(0, 3) AnnotationState.FAILED
(0, 4) Car Maintenance and Safety Discussions
(0, 5) Criticism of Government Bailouts and Corporate Ethics
(0, 6) AnnotationState.FAILED
(0, 7) Christian Theology and Philosophy Discussions
(1, 0) Baseball Stats
(1, 1) Computer Hardware and Graphics
(1, 2) Criticism of Government and Ethics
(1, 3) Car Maintenance and Safety Discussions


In [49]:
from pathlib import Path
import json

In [50]:
def load_multiline_json_logs(path: str | Path) -> list[dict]:
    """
    Helper function to load a JSONL file.

    Args:
        path (str | Path): Path to the JSONL file.

    Returns:
        list[dict]: List of JSON objects loaded from the file.
    """
    path = Path(path)
    records = []

    with path.open("r", encoding="utf-8") as f:
        for i, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError as e:
                print(f"Skipping bad JSON on line {i}: {e}")
    return records



In [51]:
debug_logs = load_multiline_json_logs(debug_logger_file)

In [52]:
for log in debug_logs:
    print(log["raw_response"])

{"baseball": "Baseball Statistics Analysis", "topic_specificity": 0.9}
{"baseball statistics": 0.8, "sports analysis": 0.9, "sports history": 0.95}
{"baseball": "baseball", "topic_specificity": 1.0}
{"Graphics": 0.95, "Computers": 0.85, "BBS": 0.8}
{"topic_name": "Computer Hardware and Graphics", "topic_specificity": 0.9}
{"topic_name": "Cryptography", "topic_specificity": 0.9}
{"Medical Discussion": 0.85, "topic_specificity": 0.85}
{"Medical Discussion": 0.85}
{"Medical Discussion": 0.8, "topic_specificity": 0.8}
{"topic_name": "Car Maintenance and Safety Discussions", "topic_specificity": 0.85}
{"topic_name": "Criticism of Government Bailouts and Corporate Ethics", "topic_specificity": 0.95}
{"Middle East Conflict": 0.85}
{"Middle East Conflict": 0.85}
{"Middle East Politics": 0.85}
{"Christian Theology": 0.85, "Christianity": 0.92, "Philosophy of Religion": 0.78}
{"Christian Theology": 0.85, "Christianity Discussion": 0.92, "Theology Debate": 0.88}
{"topic_name": "Christian Theology